# Global Population Change (2010–2020)

A cartographic workflow to visualize global population change using the **Gridded Population of the World (GPW) v4** dataset. The final output is a publication-ready thematic map in **Equal Earth projection** with official India boundary from the Survey of India.

![Final Output](output/population_change_equal_earth.png)

## Dataset

- **Source**: [GPW v4 – Population Count, Rev. 11](https://sedac.ciesin.columbia.edu/data/set/gpw-v4-population-count-rev11/data-download) (CIESIN, Columbia University)
- **Resolution**: 2.5 arc-minutes (~5 km)
- **Years**: 2010 and 2020
- **India Boundary**: [Survey of India (SOI) official shapefile](https://github.com/somdeepkundu/geoKosh/tree/main/IndiaShapes)

## Tools

`rioxarray` · `xarray-spatial` · `cartopy` · `geopandas` · `matplotlib`

---
## 1. Setup

In [ ]:
%%capture
# Install dependencies (Colab only)
if 'google.colab' in str(get_ipython()):
    !pip install rioxarray xarray-spatial cartopy geopandas

In [ ]:
import os
import zipfile
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rioxarray as rxr
import xrspatial
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.feature import ShapelyFeature

In [ ]:
# Directories
DATA_DIR = 'data'
OUTPUT_DIR = 'output'
SHP_DIR = 'india_shp'

for d in [DATA_DIR, OUTPUT_DIR, SHP_DIR]:
    os.makedirs(d, exist_ok=True)

---
## 2. Download Data

In [ ]:
def download(url, dest_folder):
    """Download a file if it doesn't already exist."""
    filepath = os.path.join(dest_folder, os.path.basename(url))
    if not os.path.exists(filepath):
        urllib.request.urlretrieve(url, filepath)
        print(f'Downloaded: {filepath}')
    else:
        print(f'Already exists: {filepath}')
    return filepath

In [ ]:
# --- Population rasters ---
GPW_BASE_URL = 'https://github.com/spatialthoughts/geopython-tutorials/releases/download/data/'

pop_2010_zip = download(GPW_BASE_URL + 'gpw-v4-population-count-rev11_2010_2pt5_min_tif.zip', DATA_DIR)
pop_2020_zip = download(GPW_BASE_URL + 'gpw-v4-population-count-rev11_2020_2pt5_min_tif.zip', DATA_DIR)

In [ ]:
# --- Official India boundary (SOI shapefile) ---
INDIA_SHP_URL = 'https://raw.githubusercontent.com/somdeepkundu/geoKosh/main/IndiaShapes/'

for ext in ['shp', 'dbf', 'prj', 'shx']:
    download(f'{INDIA_SHP_URL}IndiaBorder.{ext}', SHP_DIR)

---
## 3. Read Population Rasters

We read GeoTIFFs directly from the zip archives using `rioxarray`.

In [ ]:
def read_tif_from_zip(zip_path):
    """Extract the .tif URI from a zip file and open it with rioxarray."""
    with zipfile.ZipFile(zip_path) as zf:
        tif_name = [f for f in zf.namelist() if f.endswith('.tif')][0]
    uri = f'zip://{zip_path}!{tif_name}'
    return rxr.open_rasterio(uri, mask_and_scale=True, chunks='auto')

In [ ]:
pop2010 = read_tif_from_zip(pop_2010_zip)
pop2020 = read_tif_from_zip(pop_2020_zip)

print(f'Shape: {pop2020.shape}  |  CRS: {pop2020.rio.crs}')
pop2020

---
## 4. Compute Population Change

In [ ]:
change = pop2020 - pop2010

### 4a. Quick Look — Continuous Change

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))

change.sel(band=1).plot.imshow(
    ax=ax, vmin=-100, vmax=1000, center=0,
    add_colorbar=True, cmap='Spectral_r')

ax.set_title('Population Count Change (2010–2020)')
ax.set_axis_off()
plt.show()

---
## 5. Reclassify into Discrete Categories

| Class | Range | Label |
|-------|-------|-------|
| 1 | ≤ −100 | Decline |
| 2 | −100 to 100 | Neutral |
| 3 | 100 to 1 000 | Growth |
| 4 | > 1 000 | High Growth |

In [ ]:
CLASS_BINS = [-100, 100, 1000, np.inf]
CLASS_VALUES = [1, 2, 3, 4]

change_class = xrspatial.classify.reclassify(
    change.sel(band=1), bins=CLASS_BINS, new_values=CLASS_VALUES)

### 5a. Quick Look — Classified

In [ ]:
COLORS = ['#3288bd', '#d9d5cc', '#fdae61', '#d7191c']
LABELS = ['Decline', 'Neutral', 'Growth', 'High Growth']

levels = np.linspace(
    min(CLASS_VALUES), max(CLASS_VALUES) + 1, len(CLASS_VALUES) + 1)

fig, ax = plt.subplots(figsize=(15, 7))

change_class.plot.imshow(
    ax=ax, add_colorbar=False,
    levels=levels, vmin=1, colors=COLORS)

patches = [mpatches.Patch(color=COLORS[i], label=LABELS[i]) for i in range(4)]
ax.legend(handles=patches)
ax.set_title('Population Change Classes (2010–2020)')
ax.set_axis_off()
plt.show()

---
## 6. Export Raster Outputs

In [ ]:
# GeoTIFF in original CRS (EPSG:4326)
change_class.rio.to_raster(
    os.path.join(OUTPUT_DIR, 'change_class.tif'), compress='LZW')

# GeoTIFF reprojected to Equal Earth
change_class_ee = change_class.rio.reproject('ESRI:54035')
change_class_ee.rio.to_raster(
    os.path.join(OUTPUT_DIR, 'change_class_equal_earth.tif'), compress='LZW')

print('Rasters saved.')

---
## 7. Publication-Ready Map

Final cartographic output using **Equal Earth projection**, country boundaries, and official India border (SOI shapefile).

In [ ]:
# ── Style ──
plt.rcParams['font.family'] = 'DejaVu Sans'

COLORS = ['#3288bd', '#d9d5cc', '#fdae61', '#d7191c']
LABELS = ['Decline', 'Neutral', 'Growth', 'High Growth']
levels = np.linspace(
    min(CLASS_VALUES), max(CLASS_VALUES) + 1, len(CLASS_VALUES) + 1)

# ── Figure ──
fig, ax = plt.subplots(1, 1, subplot_kw={'projection': ccrs.EqualEarth()})
fig.set_size_inches(16, 8)
fig.patch.set_facecolor('#f9f6f1')

# ── Raster layer ──
change_class.plot.imshow(
    ax=ax,
    add_colorbar=False,
    levels=levels, vmin=1,
    colors=COLORS,
    transform=ccrs.PlateCarree()
)

# ── Boundaries ──
ax.coastlines(linewidth=0.3, color='#555555')
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor='#777777')

# Official India boundary (SOI)
india_gdf = gpd.read_file(os.path.join(SHP_DIR, 'IndiaBorder.shp'))
ax.add_feature(ShapelyFeature(
    india_gdf.geometry, ccrs.PlateCarree(),
    facecolor='none', edgecolor='#333333',
    linewidth=1.0, zorder=10
))

ax.set_global()

# ── Legend ──
patches = [mpatches.Patch(color=COLORS[i], label=LABELS[i]) for i in range(4)]
ax.legend(
    handles=patches, loc='upper right',
    frameon=True, framealpha=0.9, edgecolor='#cccccc',
    fontsize=10, title='Category', title_fontsize=11, fancybox=False
)

# ── Title (top) ──
ax.set_title(
    'Population Change Classes (2010–2020)',
    fontsize=20, fontweight='bold', color='#2d2d2d', pad=15
)

# ── Attribution (bottom) ──
fig.text(0.5, 0.04, 'Mapped by Somdeep Kundu \u00b7 4th April 2026',
         ha='center', fontsize=10, color='#888888', style='italic')
fig.text(0.5, 0.01, 'RuDRA Lab, C-TARA, IIT Bombay',
         ha='center', fontsize=9, color='#999999', style='italic')

# ── Cleanup ──
ax.set_axis_off()
ax.spines['geo'].set_edgecolor('#aaaaaa')
ax.spines['geo'].set_linewidth(0.8)

plt.tight_layout(rect=[0, 0.06, 1, 1])

# ── Save ──
fig.savefig(
    os.path.join(OUTPUT_DIR, 'population_change_equal_earth.png'),
    dpi=300, bbox_inches='tight',
    facecolor=fig.get_facecolor(), pad_inches=0.3
)

plt.show()
print(f'Map saved to {OUTPUT_DIR}/')

---

**Data Credit**: Center for International Earth Science Information Network (CIESIN), Columbia University. 2018. *Gridded Population of the World, Version 4 (GPWv4): Population Count, Revision 11.* Palisades, NY: NASA SEDAC. https://doi.org/10.7927/H4JW8BX5

**India Boundary**: Survey of India (SOI) official boundary via [geoKosh](https://github.com/somdeepkundu/geoKosh)